### Nonlinearities for Shimmer Reverb

This notebook investigates the response of the nonlinear operations presented in the paper. The library for time-domain FDN processing that the original source code relies on is not yet publicly available. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import os
from src.nonlinearities import get_nl_function
from src.time_variations import get_tv_function

# Sinusoidal input
fs = 96000
T = 4
f_sine = 440*4  # Hz
num_samples = int(T * fs)
time = np.linspace(0, T, num_samples, endpoint=False)
sine_input = 0.5*np.sin(2 * np.pi * f_sine * time)

# Helper for plotting
def plot_time_spectrum_spectrogram(signal, fs, title_prefix):
    fig, axs = plt.subplots(3, 1, figsize=(12, 10))
    t = np.arange(len(signal)) / fs
    # Time domain
    axs[0].plot(t, signal)
    axs[0].set_title(f"{title_prefix} - Time Domain")
    axs[0].set_xlabel("Time (s)")
    axs[0].set_ylabel("Amplitude")
    axs[0].grid(True)
    # Spectrum
    fft_vals = np.fft.rfft(signal)
    freqs = np.fft.rfftfreq(len(signal), 1/fs)
    axs[1].plot(freqs, 20 * np.log10(np.abs(fft_vals) + 1e-12))
    axs[1].set_title(f"{title_prefix} - Spectrum")
    axs[1].set_xlabel("Frequency (Hz)")
    axs[1].set_ylabel("Magnitude (dB)")
    axs[1].set_xlim([0, fs/2])
    axs[1].axvline(f_sine, color='k', linestyle='--')
    axs[1].grid(True)
    # Spectrogram
    axs[2].specgram(signal, NFFT=1024, Fs=fs, noverlap=512, cmap='viridis')
    axs[2].set_title(f"{title_prefix} - Spectrogram")
    axs[2].set_xlabel("Time (s)")
    axs[2].set_ylabel("Frequency (Hz)")
    axs[2].set_ylim([0, fs/2])
    return axs

plot_time_spectrum_spectrogram(sine_input, fs, "Input Sine Wave")

In [ ]:
# --- Absolute Value ---
cfwr_fun = get_nl_function("cfwr", 1, extra_arg={'degree': 1, 'fade': 'const', 'fade_const': 1}, add_dc_block=True)
cfwr_output = np.zeros_like(sine_input)
for n in range(len(sine_input)):
    cfwr_output[n] = cfwr_fun[1][0](cfwr_fun[0][0](sine_input[n]).item())
    
axs = plot_time_spectrum_spectrogram(cfwr_output, fs, "CFWR")
axs[1].axvline(f_sine*2, color='k', linestyle='--', alpha=0.3)
axs[1].axvline(f_sine*4, color='k', linestyle='--', alpha=0.3)
axs[1].set_xlim([-1000, fs/2])
plt.tight_layout()
plt.show()

print(f"Input energy: {np.sum(sine_input**2):.3f}, Output energy: {np.sum(cfwr_output**2):.3f}")

In [ ]:
# --- signal dependent fractional delay ---
sdfd_fun = get_nl_function("sdfd", 1, extra_arg={'d': 1})
sdfd_output = np.zeros_like(sine_input)
for n in range(len(sine_input)):
    sdfd_output[n] = sdfd_fun[0](sine_input[n]).item()
axs = plot_time_spectrum_spectrogram(sdfd_output, fs, "Signal Dependent Fractional Delay")
axs[1].axvline(f_sine*2, color='k', linestyle='--', alpha=0.3)
axs[1].axvline(f_sine*4, color='k', linestyle='--', alpha=0.3)
axs[1].set_xlim([-1000, fs/2])

plt.tight_layout()
plt.show()

print(f"Input energy: {np.sum(sine_input**2):.3f}, Output energy: {np.sum(sdfd_output**2):.3f}")

In [ ]:
# --- Amplitude Modulation ---
mod_fun = get_tv_function("ring_mod", 1, extra_arg={'mod_freq': 100, 'mod_amp': np.sqrt(2), 'fs': fs})
mod_output = np.zeros_like(sine_input)
for n in range(len(sine_input)):
    mod_output[n] = mod_fun[0](sine_input[n])
axs = plot_time_spectrum_spectrogram(mod_output, fs, "Amplitude Modulation (10 Hz)")
axs[1].set_xlim([-1000, fs/2])
print(f"Input energy: {np.sum(sine_input**2):.3f}, Output energy: {np.sum(mod_output**2):.3f}")


In [ ]:
# --- Pitch Shifting ---
window_size = 256 * 8 # the smaller the less latency (but bad for transient sounds - smearing)
max_delay_samps = window_size * 4 + 8
pitch_fun = get_nl_function(
    "pitchshift", 
    1, 
    extra_arg={"max_delay_samps": int(max_delay_samps), "window_size": window_size, "transpose_cents": 1200, "fs": fs},
    add_dc_block=False
)
pitch_output = np.zeros_like(sine_input)
for n in range(len(sine_input)):
    pitch_output[n] = pitch_fun[0](sine_input[n])
axs = plot_time_spectrum_spectrogram(pitch_output, fs, "Pitch Shift (+2 cents)")
axs[1].axvline(f_sine*2, color='k', linestyle='--', alpha=0.3)
axs[1].set_xlim([-1000, fs/2])

In [ ]:
# --- Granular Delay ---
nl = 'granular'
buffer_size = 256 * 8 
grain_dur_samps = 256 * 8 * 2
granular_fun = get_nl_function(
    nl, 
    1, 
    extra_arg={"max_delay_samps": buffer_size * 8, "grain_dur_samps": grain_dur_samps, "transpose_cents": 100*12, "fs": fs})

granular_output = np.zeros_like(sine_input)
for n in range(len(sine_input)):
    granular_output[n] = granular_fun[0](sine_input[n])
axs = plot_time_spectrum_spectrogram(granular_output, fs, "Granular Delay")
axs[1].axvline(f_sine*2, color='k', linestyle='--', alpha=0.3)
axs[1].set_xlim([-1000, fs/2])

In [ ]:
# Output directory 
save_dir = "output"
os.makedirs(save_dir, exist_ok=True)

waveforms = {
    "input_sine":        sine_input,
    "cfwr":              cfwr_output,
    "sdfd":              sdfd_output,
    "ring_mod":           mod_output,
    "pitch_shift":       pitch_output,
    "granular_delay":    granular_output,
}

for name, sig_orig in waveforms.items():
    # Make a COPY to avoid modifying the original
    sig = np.copy(sig_orig)
    
    # check that the signal is in the range [-1, 1]
    max_val = np.max(np.abs(sig))
    if max_val > 1:
        print(f"Warning: Signal '{name}' has max amplitude {max_val:.2f} which exceeds 1. Normalizing.")
        sig = sig / max_val
    
    path = os.path.join(save_dir, f"{name}.wav")
    sf.write(path, sig, fs, subtype='DOUBLE')
    print(f"Saved: {path}")
